In [1]:
import sys
import os 
from tqdm import tqdm
from rasterio.enums import Resampling
import glob

PROJECT_ROOT = os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print("Project root added:", PROJECT_ROOT)


Project root added: c:\Users\debas\OneDrive\Desktop\Final Year Project\Main_prj


In [2]:

#utilites

from utils.masking import apply_scl_mask
from utils.stacking import stack_bands, save_scene
from utils.resampling import resample_band, read_band, save_band
from utils.normalize import  normalize
from utils.patching import extract_and_save_patches_streaming


In [3]:
# data path
DRIVE_DIR = r"G:\My Drive\Satelite Project Dataset\Soil_Classification"
DATASET_ROOT = "../Dataset/soil_classification"
LOCAL_PATCH_DIR = os.path.join(DATASET_ROOT, "patches")


os.makedirs(LOCAL_PATCH_DIR, exist_ok=True)
print("Patches will be saved in:", LOCAL_PATCH_DIR)

Patches will be saved in: ../Dataset/soil_classification\patches


In [4]:
def find_band(scene_path, band_key):
    """
    band_key examples:
    'B02_10m', 'B03_10m', 'B04_10m', 'B08_10m',
    'B11_20m', 'SCL_20m'
    """
    files = glob.glob(os.path.join(scene_path, f"*_{band_key}.jp2"))
    
    if len(files) == 0:
        raise FileNotFoundError(f"No file found for {band_key} in {scene_path}")
    
    return files[0]  # Sentinel-2 gives exactly one match

In [5]:
def proccess_scene(scene_name):
    scene_path = os.path.join(DRIVE_DIR, scene_name)
    print(f"\nProcessing scene: {scene_name}")

    # find band files
    b02_path = find_band(scene_path, 'B02_10m')
    b03_path = find_band(scene_path, "B03_10m")
    b04_path = find_band(scene_path, "B04_10m")
    b08_path = find_band(scene_path, "B08_10m")
    b11_path = find_band(scene_path, "B11_20m")
    scl_path = find_band(scene_path, "SCL_20m")

    # read 10m bands
    b02, ref_profile = read_band(b02_path)
    b03, _ = read_band(b03_path)
    b04, _ = read_band(b04_path)
    b08, _ = read_band(b08_path)

    # read 20m bands
    b11_20, b11_profile = read_band(b11_path)
    scl_20, scl_profile = read_band(scl_path)

    # resample to 10m
    b11_10 = resample_band(b11_20, b11_profile, ref_profile, Resampling.bilinear)
    scl_10 = resample_band(scl_20, scl_profile, ref_profile, Resampling.nearest)

    # apply SCL mask
    b02 = apply_scl_mask(b02, scl_10)
    b03 = apply_scl_mask(b03, scl_10)
    b04 = apply_scl_mask(b04, scl_10)
    b08 = apply_scl_mask(b08, scl_10)

    # STREAMING PATCH EXTRACTION (KEY LINE)
    scene_patch_dir = os.path.join(LOCAL_PATCH_DIR, scene_name)

    extract_and_save_patches_streaming(
        b04=b04,
        b03=b03,
        b02=b02,
        b08=b08,
        scl=scl_10,
        patch_size=128,
        out_dir=scene_patch_dir,
        scene_id=scene_name
    )

    # free memory explicitly
    del b02, b03, b04, b08, scl_10, b11_20





In [6]:
test_scene = os.listdir(DRIVE_DIR)[0]
proccess_scene(test_scene)



Processing scene: scene_1
Saved 1968 patches for scene_1


In [8]:
import numpy as np

scene_patch_dir = os.path.join(LOCAL_PATCH_DIR, test_scene)
files = os.listdir(scene_patch_dir)

print("Number of patches:", len(files))

sample = np.load(os.path.join(scene_patch_dir, files[0]))
print("Sample shape:", sample.shape)
print("Value range:", sample.min(), sample.max())

Number of patches: 1968
Sample shape: (128, 128, 4)
Value range: 0.0 4.26e+03


In [9]:
scene_names = [
    s for s in os.listdir(DRIVE_DIR)
    if os.path.isdir(os.path.join(DRIVE_DIR, s))
]

print("Total scenes found:", len(scene_names))

for scene_name in tqdm(scene_names):
    proccess_scene(scene_name)


Total scenes found: 11


  0%|          | 0/11 [00:00<?, ?it/s]


Processing scene: scene_1


  9%|▉         | 1/11 [01:18<13:06, 78.61s/it]

Saved 1968 patches for scene_1

Processing scene: scene_2


...

Saved 5532 patches for scene_8

Processing scene: scene_7
Saved 4851 patches for scene_7


 91%|█████████ | 10/11 [21:30<02:19, 139.12s/it]


Processing scene: scene_11


100%|██████████| 11/11 [24:00<00:00, 130.92s/it]

Saved 5076 patches for scene_11
